# 05b — Genuine Source / Script Hold-Out Training
**BanglaCyberBench** | Resolves the robustness leakage (Issue 5)

**Why this notebook exists.** NB08 previously evaluated the *random-split* checkpoints (`models_main/`) on each held-out source. But those checkpoints were fine-tuned on the random 80% split, which **contains** samples from every source — so the held-out source was already seen during fine-tuning. That is **not** a real hold-out and inflates the robustness numbers.

**The fix.** For each of the 5 hold-out settings (4 source + 1 script), we fine-tune dedicated models on that setting's `*_train.csv` (which **excludes** the held-out source/script) and later evaluate on its `*_test.csv` in NB08. The held-out data is now genuinely unseen during fine-tuning.

**Protocol (locked):** 3 backbones × 3 seeds × 5 hold-outs = **45 runs**.

**Power-surge safety:** every epoch caches `last_ckpt.pt`; completed runs are skipped on restart, and an interrupted run resumes from its last cached epoch.

**Run AFTER NB03 (splits regenerated). Independent of NB05 main runs.**

In [ ]:
# ── Install — Python 3.13 compatible ───────────────────────────────
# tokenizers >= 0.19.0 ships cp313 wheels on PyPI.
# transformers >= 4.44.0 requires tokenizers >= 0.19.
import subprocess, sys

print(f"Python: {sys.version}")

result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "tokenizers>=0.19.0",
     "transformers>=4.44.0",
     "accelerate>=0.34.0",
     "sentencepiece",
     "--upgrade", "--quiet"],
    capture_output=True, text=True, check=False
)
if result.returncode != 0:
    print("pip stderr:", result.stderr[-400:])
else:
    print("pip install: OK")

import torch, tokenizers, transformers
print(f"torch       : {torch.__version__}")
print(f"tokenizers  : {tokenizers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU         : {p.name}  {p.total_memory/1e9:.1f} GB")
print("✅ Environment OK")


In [ ]:
import os, json, time, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True   # speeds up Ampere/Ada GPUs
    torch.backends.cudnn.allow_tf32  = True
    torch.backends.cudnn.benchmark   = True

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)

def infer_num_workers():
    n = os.cpu_count() or 2
    return min(4, max(2, n // 2)) if os.name == "nt" else min(6, max(2, n // 2))

print(f"num_workers: {infer_num_workers()}")


## Config — hold-out training

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONFIG — Hold-out training (mirrors NB05 main config; uniform LR)
# ═══════════════════════════════════════════════════════════════════
CONFIG = {
    "models": {
        "banglabert": "csebuetnlp/banglabert",
        "muril":      "google/muril-base-cased",
        "xlmr":       "xlm-roberta-base",
    },
    "max_length":       128,
    "batch_size":       16,
    "eval_batch_size":  32,
    "grad_accum_steps": 2,
    "num_workers":      0,
    "epochs":           8,
    "encoder_lr":       2e-5,
    "head_lr":          8e-5,
    "weight_decay":     0.01,
    "warmup_ratio":     0.10,
    "lr_decay_factor":  0.90,
    "max_grad_norm":    1.0,
    "label_smoothing":        0.03,
    "focal_gamma_binary":     1.5,
    "focal_gamma_abuse_type": 3.0,   # higher: focus learning on hard abuse classes
    "class_weight_beta":      0.999,
    "dropout":          0.25,
    "head_hidden_dim":  384,
    "patience":    3,
    "use_fp16":    torch.cuda.is_available(),
    "use_compile": False,
}

# Final design = uniform LR (no layer-wise decay), consistent with NB05.
USE_LRDECAY = False

# SINGLE-TASK: abuse-type only (binary head removed).
TASK_CONFIG = {
    "abuse_type": {"col": "label_type",   "num_classes": None, "loss_weight": 1.00, "monitor_weight": 1.00},
}

SPLIT_DIR    = "../data/splits"
HOLDOUT_ROOT = "../outputs/models_holdout"   # one subdir per hold-out setting
os.makedirs(HOLDOUT_ROOT, exist_ok=True)

RUN_MODELS = ["banglabert", "muril", "xlmr"]
RUN_SEEDS  = [42, 123, 456]
FORCE_RETRAIN = False

# The 5 hold-out settings. Each maps to its train/val split prefix from NB03.
# (The matching *_test.csv is evaluated later in NB08.)
HOLDOUTS = {
    "source_holdout_banth":            {"train": "source_holdout_banth_train.csv",
                                        "val":   "source_holdout_banth_val.csv",
                                        "exclude_source": "banth"},
    "source_holdout_bd_shs":           {"train": "source_holdout_bd_shs_train.csv",
                                        "val":   "source_holdout_bd_shs_val.csv",
                                        "exclude_source": "bd_shs"},
    "source_holdout_facebook_44001":   {"train": "source_holdout_facebook_44001_train.csv",
                                        "val":   "source_holdout_facebook_44001_val.csv",
                                        "exclude_source": "facebook_44001"},
    "source_holdout_multilabel_12557": {"train": "source_holdout_multilabel_12557_train.csv",
                                        "val":   "source_holdout_multilabel_12557_val.csv",
                                        "exclude_source": "multilabel_12557"},
    "script_holdout_romanized":        {"train": "script_holdout_train.csv",
                                        "val":   "script_holdout_val.csv",
                                        "exclude_script": "romanized"},
}

print(f"Hold-out settings : {list(HOLDOUTS.keys())}")
print(f"Per setting        : {len(RUN_MODELS)} models × {len(RUN_SEEDS)} seeds = {len(RUN_MODELS)*len(RUN_SEEDS)} runs")
print(f"Total runs         : {len(HOLDOUTS)*len(RUN_MODELS)*len(RUN_SEEDS)}")
print(f"USE_LRDECAY        : {USE_LRDECAY}")


## Label consolidation (89 → 9) + per-hold-out encoder builder

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LABEL CONSOLIDATION  89 raw → 9 canonical
# Priority: threat > sexual > religious > gender > political >
#           abusive > personal > other > none
# ═══════════════════════════════════════════════════════════════════

TYPE_MAP = {
    # none
    "none": "none", "not bully": "none",
    # threat  (highest priority)
    "threat": "threat", "threat,spam": "threat",
    "callToViolence": "threat", "callToViolence_slander": "threat",
    "callToViolence_gender": "threat", "callToViolence_religion": "threat",
    "callToViolence_religion_slander": "threat",
    "callToViolence_gender_religion_slander": "threat",
    "callToViolence_gender_slander": "threat",
    "religious,threat": "threat", "sexual,threat": "threat",
    "sexual,religious,threat": "threat",
    # sexual   (sexual > religious — FIXED from v2 which had sexual,religious→religious)
    "sexual": "sexual",
    "sexual,religious": "sexual",   # ← FIX: sexual beats religious
    "sexual,spam": "sexual",
    # religious
    "religious": "religious", "Religious": "religious",
    "religion": "religious", "religious,spam": "religious",
    "religion_slander": "religious",
    "gender_religion": "religious",   # religious > gender
    "gender_religion_slander": "religious",
    # gender
    "gender": "gender", "Gender": "gender", "gender_slander": "gender",
    # political
    "Political": "political",
    # personal
    "Personal Offense": "personal", "Body Shaming": "personal",
    "Origin": "personal", "slander": "personal", "Misc": "personal",
    # abusive
    "Abusive/Violence": "abusive", "troll": "abusive",
    # other
    "spam": "other",
}

PRIORITY = ["threat","sexual","religious","gender","political","abusive","personal","other","none"]

def consolidate_type(val):
    if not isinstance(val, str) or not val.strip():
        return "none"
    val = val.strip()
    if val in TYPE_MAP:
        return TYPE_MAP[val]
    # Split compound labels (comma-separated), resolve each part, pick highest priority
    parts = [p.strip() for p in val.replace(";", ",").split(",")]
    cands = [TYPE_MAP[p] for p in parts if p in TYPE_MAP]
    if cands:
        for pc in PRIORITY:
            if pc in cands:
                return pc
    # Substring fallback (covers unseen compound variants)
    sl = val.lower().replace("_", " ")
    for kw, cls in [("threat","threat"), ("calltoviolence","threat"),
                    ("sexual","sexual"), ("religious","religious"),
                    ("religion","religious"), ("gender","gender"),
                    ("political","political"), ("abusive","abusive"),
                    ("violence","abusive"), ("personal","personal"),
                    ("slander","personal"), ("origin","personal"),
                    ("body","personal"), ("misc","personal"),
                    ("spam","other")]:
        if kw in sl.replace(" ",""):
            return cls
    return "other"   # safe fallback for any unrecognised label


def to_scalar(x):
    return x.item() if hasattr(x, "item") else x

def make_jsonable(obj):
    if isinstance(obj, dict):  return {str(k): make_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list,tuple)): return [make_jsonable(x) for x in obj]
    if isinstance(obj, np.integer):   return int(obj)
    if isinstance(obj, np.floating):  return float(obj)
    return obj

def build_label_encoders(train_df, val_df, task_config):
    """Build label encoders from train+val of a SINGLE hold-out setting.
    Returns (label_encoders, active_tasks). abuse_type is forced to the full
    9-class space so heads are identical across hold-outs and match NB05/NB08."""
    CANON9 = ["abusive","gender","none","other","personal","political","religious","sexual","threat"]
    label_encoders, active_tasks = {}, {}
    import copy
    tc = copy.deepcopy(task_config)
    for task_name, task_cfg in tc.items():
        col = task_cfg["col"]
        if col not in train_df.columns:
            continue
        if task_name == "abuse_type":
            enc = {v: i for i, v in enumerate(CANON9)}   # fixed 9-class space
        else:
            all_vals = sorted(set(to_scalar(v)
                for v in pd.concat([train_df[col], val_df[col]]).dropna()))
            enc = {v: i for i, v in enumerate(all_vals)}
        label_encoders[task_name] = enc
        task_cfg["num_classes"]   = len(enc)
        active_tasks[task_name]   = task_cfg
    return label_encoders, active_tasks

print("Consolidation + label-encoder helpers defined ✅")


## Dataset, model, loss (identical to NB05)

In [ ]:
class CyberBullyDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, active_tasks, label_encoders, text_col):
        self.texts  = df.reset_index(drop=True)[text_col].fillna("").astype(str).tolist()
        self.tok    = tokenizer
        self.maxlen = max_length
        self.labels = {}
        for tname, tcfg in active_tasks.items():
            enc  = label_encoders[tname]
            self.labels[tname] = [
                int(enc.get(to_scalar(v), -1))
                for v in df[tcfg["col"]].fillna("__missing__")
            ]

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.maxlen,
                       truncation=True, padding=False)
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        for t in self.labels:
            item[f"label_{t}"] = torch.tensor(self.labels[t][idx], dtype=torch.long)
        return item


class MultiTaskCollator:
    def __init__(self, tokenizer): self.tokenizer = tokenizer
    def __call__(self, features):
        lkeys  = [k for k in features[0] if k.startswith("label_")]
        labels = {k: torch.stack([f[k] for f in features]) for k in lkeys}
        tfeats = [{k: v for k, v in f.items() if not k.startswith("label_")} for f in features]
        batch  = self.tokenizer.pad(tfeats, padding=True, return_tensors="pt")
        batch.update(labels)
        return batch

print("Dataset + Collator defined ✅")


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma, self.weight, self.ls = gamma, weight, label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             reduction="none", label_smoothing=self.ls)
        loss = ((1 - torch.exp(-ce)) ** self.gamma) * ce
        return loss.mean()


class TaskHead(nn.Module):
    def __init__(self, hidden, n_cls, dropout=0.25, inner=384):
        super().__init__()
        inner = min(inner, hidden)
        self.net = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden, inner),
            nn.GELU(), nn.LayerNorm(inner),
            nn.Dropout(dropout), nn.Linear(inner, n_cls),
        )
    def forward(self, x): return self.net(x)


class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, active_tasks, dropout=0.25, head_hidden_dim=384):
        super().__init__()
        self.encoder  = AutoModel.from_pretrained(model_name)
        hidden        = self.encoder.config.hidden_size
        mtype         = self.encoder.config.model_type.lower()
        self._use_tti = mtype not in ("roberta", "xlm-roberta")  # XLM-R has no TTI
        self.heads    = nn.ModuleDict({
            t: TaskHead(hidden, cfg["num_classes"], dropout, head_hidden_dim)
            for t, cfg in active_tasks.items()
        })

    def _pool(self, out, mask):
        hs   = out.last_hidden_state
        cls  = hs[:, 0]
        mean = (hs * mask.unsqueeze(-1).float()).sum(1) / mask.sum(1, keepdim=True).float().clamp(1)
        return 0.5 * cls + 0.5 * mean       # CLS + mean-pool blend

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if self._use_tti and token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids
        out = self.encoder(**kw)
        p   = self._pool(out, attention_mask)
        return {t: h(p) for t, h in self.heads.items()}

print("Model classes defined ✅")


In [ ]:
def get_layerwise_params(model, enc_lr, head_lr, decay, wd):
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    n_layers = 0
    for name, _ in model.encoder.named_parameters():
        for p in name.split("."):
            if p.isdigit(): n_layers = max(n_layers, int(p)+1)
    n_layers = max(n_layers, 12)

    groups = []
    for name, param in model.encoder.named_parameters():
        if not param.requires_grad: continue
        ln = 0
        for p in name.split("."):
            if p.isdigit(): ln = int(p); break
        lr = enc_lr * (decay ** (n_layers - ln - 1))
        _wd = 0.0 if any(nd in name for nd in no_decay) else wd
        groups.append({"params": [param], "lr": lr, "weight_decay": _wd})
    for name, param in model.heads.named_parameters():
        if not param.requires_grad: continue
        _wd = 0.0 if any(nd in name for nd in no_decay) else wd
        groups.append({"params": [param], "lr": head_lr, "weight_decay": _wd})
    return groups


def get_uniform_params(model, enc_lr, head_lr, wd):
    """No layer-wise decay: encoder params all at enc_lr, head params at head_lr.
    This is the FINAL design (USE_LRDECAY=False). Weight decay is skipped for
    bias/LayerNorm terms exactly as in the layer-wise path."""
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    groups = []
    enc_decay, enc_nodecay = [], []
    for name, param in model.encoder.named_parameters():
        if not param.requires_grad: continue
        (enc_nodecay if any(nd in name for nd in no_decay) else enc_decay).append(param)
    groups.append({"params": enc_decay,   "lr": enc_lr, "weight_decay": wd})
    groups.append({"params": enc_nodecay, "lr": enc_lr, "weight_decay": 0.0})
    head_decay, head_nodecay = [], []
    for name, param in model.heads.named_parameters():
        if not param.requires_grad: continue
        (head_nodecay if any(nd in name for nd in no_decay) else head_decay).append(param)
    groups.append({"params": head_decay,   "lr": head_lr, "weight_decay": wd})
    groups.append({"params": head_nodecay, "lr": head_lr, "weight_decay": 0.0})
    return groups


def build_class_weights(series, enc, beta=0.999, max_w=10.0):
    # Effective-sample-number weights, clamped so focal loss never NaNs.
    mapped = series.map(enc).dropna().astype(int)
    n_cls  = len(enc)
    counts = mapped.value_counts().sort_index()
    w      = np.ones(n_cls, dtype=np.float32)
    for i in range(n_cls):
        n = int(counts.get(i, 0))
        if n > 0:
            eff = (1.0 - beta**n) / max(1.0 - beta, 1e-12)
            w[i] = 1.0 / max(eff, 1e-9)
    # Clip: never let any class weight exceed max_w × min weight
    w = np.clip(w, w.min(), w.min() * max_w)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

print("Optimizer (layerwise + uniform) + weight helpers defined ✅")


In [ ]:
@torch.no_grad()
def evaluate_tasks(model, loader, active_tasks, label_encoders):
    model.eval()
    preds  = {t: [] for t in active_tasks}
    labels = {t: [] for t in active_tasks}
    for batch in loader:
        batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids"))
        for t in active_tasks:
            lbl  = batch[f"label_{t}"].cpu().numpy()
            pred = logits[t].argmax(-1).cpu().numpy()
            m    = lbl >= 0
            preds[t].extend(pred[m]); labels[t].extend(lbl[m])
    out = {}
    for t in active_tasks:
        y, p = np.array(labels[t]), np.array(preds[t])
        out[t] = {
            "macro_f1":    round(float(f1_score(y, p, average="macro",    zero_division=0)), 4),
            "weighted_f1": round(float(f1_score(y, p, average="weighted", zero_division=0)), 4),
            "accuracy":    round(float(accuracy_score(y, p)), 4),
            "mcc":         round(float(matthews_corrcoef(y, p)), 4),
        }
    return out


@torch.no_grad()
def get_logits(model, loader, active_tasks):
    model.eval()
    buf = {t: [] for t in active_tasks}
    for batch in loader:
        batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids"))
        for t in active_tasks: buf[t].append(logits[t].cpu())
    return {t: torch.cat(v, 0) for t, v in buf.items()}

print("Evaluation helpers defined ✅")


## Hold-out training function (with resume + per-epoch caching)

In [ ]:
def train_holdout_run(model_key, model_name, save_dir,
                      train_df, val_df, active_tasks, label_encoders, config, seed, text_col):
    """Train one (model, seed) on a hold-out TRAIN split. Saves best_model.pt +
    val_logits.pt. Resumes from last_ckpt.pt if a previous attempt was interrupted."""
    set_seed(seed)
    os.makedirs(save_dir, exist_ok=True)

    tok      = AutoTokenizer.from_pretrained(model_name)
    collator = MultiTaskCollator(tok)
    lkw = dict(collate_fn=collator, num_workers=config["num_workers"],
               pin_memory=torch.cuda.is_available(),
               persistent_workers=config["num_workers"]>0)

    train_loader = DataLoader(CyberBullyDataset(train_df, tok, config["max_length"],
                              active_tasks, label_encoders, text_col),
                              batch_size=config["batch_size"], shuffle=True, drop_last=True, **lkw)
    val_loader   = DataLoader(CyberBullyDataset(val_df, tok, config["max_length"],
                              active_tasks, label_encoders, text_col),
                              batch_size=config["eval_batch_size"], shuffle=False, **lkw)

    model = MultiTaskTransformer(model_name, active_tasks,
                                 dropout=config["dropout"],
                                 head_hidden_dim=config["head_hidden_dim"]).to(device)

    criteria = {}
    for tname, tcfg in active_tasks.items():
        w     = build_class_weights(train_df[tcfg["col"]], label_encoders[tname],
                                    beta=config["class_weight_beta"])
        gamma = config.get(f"focal_gamma_{tname}", 2.0)
        criteria[tname] = FocalLoss(gamma=gamma, weight=w, label_smoothing=config["label_smoothing"])

    if USE_LRDECAY:
        pg = get_layerwise_params(model, config["encoder_lr"], config["head_lr"],
                                  config["lr_decay_factor"], config["weight_decay"])
    else:
        pg = get_uniform_params(model, config["encoder_lr"], config["head_lr"], config["weight_decay"])
    optimizer = torch.optim.AdamW(pg)
    n_steps   = math.ceil(len(train_loader)/config["grad_accum_steps"]) * config["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, int(n_steps*config["warmup_ratio"]), n_steps)
    scaler    = torch.amp.GradScaler("cuda") if (config["use_fp16"] and device.type=="cuda") else None

    # ── Resume support ──────────────────────────────────────────────
    start_epoch, best_score, no_improve = 0, -1.0, 0
    last_ckpt = f"{save_dir}/last_ckpt.pt"
    if not FORCE_RETRAIN and os.path.exists(last_ckpt):
        try:
            ck = torch.load(last_ckpt, map_location=device, weights_only=False)
            model.load_state_dict(ck["model_state"])
            start_epoch = ck["epoch"] + 1
            best_score  = ck["best_score"]; no_improve = ck["no_improve"]
            print(f"    ▶ Resumed from epoch {start_epoch} (best={best_score:.4f})")
        except Exception as e:
            print(f"    ⚠ Could not resume ({e}); starting fresh.")

    for epoch in range(start_epoch, config["epochs"]):
        model.train(); optimizer.zero_grad(set_to_none=True)
        ep_loss, t0 = 0.0, time.time()
        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            with torch.autocast(device_type=device.type,
                                enabled=config["use_fp16"] and device.type=="cuda"):
                logits = model(batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids"))
                loss = torch.tensor(0.0, device=device)
                for tname, tcfg in active_tasks.items():
                    lbl = batch[f"label_{tname}"]; vm = lbl >= 0
                    if vm.any():
                        tl = criteria[tname](logits[tname][vm], lbl[vm])
                        if not (torch.isnan(tl) or torch.isinf(tl)):
                            loss = loss + tcfg["loss_weight"] * tl
                loss = loss / config["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if step % config["grad_accum_steps"] == 0 or step == len(train_loader):
                if scaler:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
                    scaler.step(optimizer); scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
                    optimizer.step()
                scheduler.step(); optimizer.zero_grad(set_to_none=True)
            ep_loss += loss.item() * config["grad_accum_steps"]
        avg_loss = ep_loss / max(len(train_loader), 1)

        vm = evaluate_tasks(model, val_loader, active_tasks, label_encoders)
        abu_f1 = vm.get("abuse_type", {}).get("macro_f1", 0.0)
        monitor = abu_f1   # SINGLE-TASK

        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "best_score": best_score, "no_improve": no_improve}, last_ckpt)
        flag = ""
        if monitor > best_score:
            best_score = monitor; no_improve = 0
            torch.save(model.state_dict(), f"{save_dir}/best_model.pt"); flag = " ✅BEST"
        else:
            no_improve += 1
        print(f"    Ep{epoch+1:2}/{config['epochs']} loss={avg_loss:.4f} "
              f"bin={bin_f1:.4f} abu={abu_f1:.4f} mon={monitor:.4f} "
              f"{(time.time()-t0)/60:.1f}min{flag}")
        if no_improve >= config["patience"]:
            print(f"    Early stop ep{epoch+1}"); break

    # Final: load best, save val logits (for any later use), write results.json
    model.load_state_dict(torch.load(f"{save_dir}/best_model.pt", map_location=device, weights_only=True))
    val_metrics = evaluate_tasks(model, val_loader, active_tasks, label_encoders)
    torch.save(get_logits(model, val_loader, active_tasks), f"{save_dir}/val_logits.pt")
    with open(f"{save_dir}/results.json", "w") as f:
        json.dump(make_jsonable({"model": model_key, "seed": seed,
                                 "best_val_monitor_score": round(best_score,4),
                                 "val_metrics": val_metrics}), f, indent=2)
    del model; torch.cuda.empty_cache()
    return val_metrics

def holdout_is_valid(save_dir):
    p = f"{save_dir}/best_model.pt"
    if not os.path.exists(p): return False
    try: torch.load(p, map_location="cpu", weights_only=True); return True
    except: return False

print("Hold-out training function defined ✅")


## Run all hold-out training

For each setting we load its `_train`/`_val` split (held-out source/script **absent**), assert the leakage-prevention property, then train 3×3 models. Completed runs are skipped; interrupted runs resume.

In [ ]:
def consolidate_split(df_):
    """Apply the 89→9 consolidation to a split's raw label_type (same as NB05)."""
    df_ = df_.copy()
    df_["label_type"] = df_["label_type"].apply(consolidate_type)
    return df_

for ho_name, ho in HOLDOUTS.items():
    print("\n" + "#"*70)
    print(f"#  HOLD-OUT: {ho_name}")
    print("#"*70)

    tr_path = os.path.join(SPLIT_DIR, ho["train"])
    va_path = os.path.join(SPLIT_DIR, ho["val"])
    if not (os.path.exists(tr_path) and os.path.exists(va_path)):
        print(f"  ⚠ Split files missing for {ho_name} — run NB03 first. Skipping.")
        continue

    tr_df = pd.read_csv(tr_path); va_df = pd.read_csv(va_path)
    text_col = "text_clean" if "text_clean" in tr_df.columns else "text"
    for d in (tr_df, va_df):
        d[text_col] = d[text_col].fillna("").astype(str)

    # ── LEAKAGE GUARD: the held-out source/script must NOT appear in train/val ──
    if "exclude_source" in ho and "source" in tr_df.columns:
        leaked = (tr_df["source"] == ho["exclude_source"]).sum() + \
                 (va_df["source"] == ho["exclude_source"]).sum()
        assert leaked == 0, (f"LEAKAGE: {leaked} rows of held-out source "
                             f"'{ho['exclude_source']}' found in train/val!")
        print(f"  ✅ No-leakage verified: source '{ho['exclude_source']}' "
              f"absent from train+val ({len(tr_df):,} train / {len(va_df):,} val).")
    if "exclude_script" in ho and "script" in tr_df.columns:
        leaked = (tr_df["script"] == ho["exclude_script"]).sum() + \
                 (va_df["script"] == ho["exclude_script"]).sum()
        assert leaked == 0, (f"LEAKAGE: {leaked} rows of held-out script "
                             f"'{ho['exclude_script']}' found in train/val!")
        print(f"  ✅ No-leakage verified: script '{ho['exclude_script']}' "
              f"absent from train+val ({len(tr_df):,} train / {len(va_df):,} val).")

    # Consolidate labels + build encoders for THIS setting (9-class fixed space)
    tr_df = consolidate_split(tr_df); va_df = consolidate_split(va_df)
    label_encoders, active_tasks = build_label_encoders(tr_df, va_df, TASK_CONFIG)
    setting_dir = os.path.join(HOLDOUT_ROOT, ho_name)
    os.makedirs(setting_dir, exist_ok=True)
    with open(os.path.join(setting_dir, "label_encoders.json"), "w", encoding="utf-8") as f:
        json.dump(make_jsonable(label_encoders), f, indent=2, ensure_ascii=False)

    for model_key in RUN_MODELS:
        model_name = CONFIG["models"][model_key]
        for seed in RUN_SEEDS:
            save_dir = os.path.join(setting_dir, f"{model_key}_seed{seed}")
            if not FORCE_RETRAIN and holdout_is_valid(save_dir):
                print(f"  ⏭  {model_key} seed={seed} — already done")
                continue
            print(f"\n  ── {model_key} seed={seed} ──")
            try:
                train_holdout_run(model_key, model_name, save_dir,
                                  tr_df, va_df, active_tasks, label_encoders,
                                  CONFIG, seed, text_col)
            except Exception as e:
                import traceback
                print(f"  ❌ {model_key} seed={seed} FAILED: {e}")
                traceback.print_exc()
            torch.cuda.empty_cache()

print("\n🎉 All hold-out training complete (or resumed/skipped where already done).")


---
**Next:** Run NB08 (reworked) — it loads these per-hold-out checkpoints and evaluates each on its matching `*_test.csv`. The in-domain split still uses `models_main/`.